# 23 Konsistensi Judul dan Isi

Menilai apakah klaim sesuai dengan isi dokumen. Contoh: kalimat pertama ringkasan IndoSum sebagai judul, dengan
negatif buatan: artikel lain, dan entitas atau angka yang ditukar.

**Daftar isi**

1. Kartu Masalah
2. Kapan Pakai Notebook Ini
3. Config
4. EDA
5. Preprocessing
6. Validasi
7. Baseline
8. Model Utama
9. Post-processing
10. Post-analysis
11. Keluaran

## 1. Kartu Masalah

| Bagian | Isi |
|---|---|
| **Task** | Memeriksa apakah judul sesuai isi |
| **Unit prediksi** | Satu pasangan judul dan isi mendapat label biner |
| **Metrik** | Macro-F1 |
| **Masalah terbesar** | Negatif dibuat sendiri, jadi skor hanya berlaku untuk tiga jenis kesalahan yang disimulasikan. Perubahan angka paling sulit dideteksi. |
| **Data contoh** | IndoSum: 4.498 artikel, negatif buatan |

## 2. Kapan Pakai Notebook Ini

**Pakai kalau** soal meminta mendeteksi judul yang tidak sesuai isi, clickbait, atau klaim yang diubah. Kalau
negatifnya buatan, cari dulu **cara pembuatannya** dari data, karena itu biasanya sumber sinyal terbesar.

**Yang perlu diganti kalau datanya beda:**
- Blok pembuatan pasangan di bagian 5 tidak dipakai kalau label sudah ada.
- `make_folds`: artikel yang sama wajib satu fold.

## 3. Config

**Import dan lingkungan.** Mode offline dinyalakan di sini, dan progress bar serta warning
dimatikan supaya output tetap bersih.

In [1]:
import importlib.util
import json
import os
import random
import subprocess
import sys
import time
import warnings
from contextlib import contextmanager
from pathlib import Path

os.environ.update(
    CUDA_VISIBLE_DEVICES="0",
    HF_HUB_OFFLINE="1",
    TRANSFORMERS_OFFLINE="1",
    TOKENIZERS_PARALLELISM="false",
    TQDM_DISABLE="1",
    PYTHONWARNINGS="ignore",
    HF_HUB_DISABLE_PROGRESS_BARS="1",
)
warnings.filterwarnings("ignore")
sys.modules["tensorflow"] = None  # tidak dipakai; mencegah log TensorFlow saat umap diimpor

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402
from transformers.utils import logging as hf_logging  # noqa: E402

hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
pd.set_option("display.max_colwidth", 90, "display.max_rows", 30, "display.width", 140)
plt.rcParams.update(
    {
        "figure.figsize": (8, 3.5),
        "figure.dpi": 90,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
print(f"device={DEVICE} | torch={torch.__version__}")

device=cuda | torch=2.11.0+cu130


**Lokasi file.** `find_pack` mencari paket offline, dan `find_file` mencari file data di Kaggle
atau folder lokal. Di mesin lokal, set `PACK_DIR` dan `DATA_ROOT` lewat environment variable.

In [2]:
INPUT_ROOTS = [Path(p) for p in (os.environ.get("DATA_ROOT"), "/kaggle/input", ".") if p]


def find_pack() -> Path:
    """Cari folder paket offline lewat file pack_manifest.json."""
    if os.environ.get("PACK_DIR"):
        return Path(os.environ["PACK_DIR"])
    for root in INPUT_ROOTS:
        for manifest in root.glob("**/pack_manifest.json"):
            return manifest.parent
    raise FileNotFoundError("Paket offline tidak ketemu. Set environment variable PACK_DIR.")


def find_file(name: str) -> Path:
    """Cari file data berdasarkan nama di semua folder input."""
    for root in INPUT_ROOTS:
        for path in root.glob(f"**/{name}"):
            return path
    raise FileNotFoundError(f"File {name} tidak ketemu di {INPUT_ROOTS}")


def install_offline(*modules: str) -> None:
    """Pasang library dari wheel paket offline, hanya bila belum terpasang."""
    missing = [m for m in modules if importlib.util.find_spec(m.replace("-", "_")) is None]
    if missing:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-index",
                "--find-links",
                str(PACK_DIR / "wheels"),
                "--root-user-action=ignore",
                *missing,
            ],
            check=True,
        )


PACK_DIR = find_pack()
MODEL_DIR = PACK_DIR / "models"
EXTRA_DATA_DIR = PACK_DIR / "data"
RESULT_DIR = Path("hasil")
RESULT_DIR.mkdir(exist_ok=True)
print(f"paket offline: {PACK_DIR}")

paket offline: /kaggle/input/notebooks/ivanwllm/gemastik-nlp-00-setup-offline


**Pencatat waktu dan ringkasan.** `stage` mencatat durasi tiap tahap, dan `run_summary` mencetak
ringkasan akhir yang dibaca skrip audit.

In [3]:
RUN_START = time.time()
STAGE_SECONDS: dict[str, float] = {}


@contextmanager
def stage(name: str):
    """Catat durasi satu tahap."""
    start = time.time()
    yield
    STAGE_SECONDS[name] = round(time.time() - start, 1)
    print(f"[waktu] {name}: {STAGE_SECONDS[name]:.0f} detik")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def run_summary(task: str, n_rows: int, metrics: dict, needs_gpu: bool = True) -> None:
    """Simpan dan cetak ringkasan run."""
    total = int(time.time() - RUN_START)
    used_gpu = torch.cuda.is_available() and torch.cuda.max_memory_allocated() > 0
    summary = {
        "task": task,
        "mode": MODE,
        "use_pretrained": USE_PRETRAINED,
        "n_rows": n_rows,
        "total_detik": total,
        "gpu": used_gpu,
        "tahap_detik": STAGE_SECONDS,
        "metrik": metrics,
    }
    (RESULT_DIR / "ringkasan.json").write_text(json.dumps(summary, indent=1))
    note = "" if needs_gpu else " gpu_tidak_wajib"
    print(
        f"RINGKASAN_RUN task={task} mode={MODE} n_rows={n_rows} total_detik={total} "
        f"gpu={used_gpu}{note}"
    )
    print(json.dumps(metrics, indent=1))

**Pengaturan utama.** Semua yang perlu diubah ada di sel ini.

In [4]:
MODE = "penuh"  # "cepat" (<30 menit) atau "penuh" (<2 jam)
USE_PRETRAINED = True  # False: hanya fitur relasi, tanpa file model dari luar
SEED = 42
N_ROWS = 5000
NEGATIVE_SHARE = 0.5
MAX_LEN = 256
BODY_WORDS = 200

SETTINGS = {
    "cepat": {"folds": 3, "epochs": 2, "model_runs": [("indobert", SEED)]},
    "penuh": {"folds": 5, "epochs": 3, "model_runs": [("indobert", SEED), ("indobert", SEED + 1)]},
}[MODE]

seed_everything(SEED)
print(SETTINGS)

{'folds': 5, 'epochs': 3, 'model_runs': [('indobert', 42), ('indobert', 43)]}


**Util teks.** `clean_text` hanya membuang gangguan teknis (URL, mention, spasi) dan
mempertahankan emoji serta tanda seru karena keduanya membawa sinyal. `normalize_slang` memakai
kamus alay. `near_duplicate_groups` menemukan komentar kembar supaya tidak bocor antar fold.

In [5]:
import re  # noqa: E402

from sklearn.feature_extraction.text import TfidfVectorizer  # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.metrics import roc_auc_score  # noqa: E402
from sklearn.model_selection import cross_val_predict  # noqa: E402
from sklearn.neighbors import NearestNeighbors  # noqa: E402

URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
SPACE_RE = re.compile(r"\s+")


def clean_text(text: str) -> str:
    """Bersihkan URL, mention, dan spasi berlebih. Huruf kecil, emoji tetap."""
    text = URL_RE.sub(" ", str(text))
    text = MENTION_RE.sub(" ", text)
    return SPACE_RE.sub(" ", text).strip().lower()


def load_slang_dict() -> dict[str, str]:
    """Muat kamus kata informal ke kata baku."""
    path = next((EXTRA_DATA_DIR / "kamus_alay").glob("**/colloquial-indonesian-lexicon.csv"))
    lexicon = pd.read_csv(path)
    return dict(zip(lexicon["slang"].astype(str), lexicon["formal"].astype(str)))


def normalize_slang(text: str, slang: dict[str, str]) -> str:
    """Ganti kata informal dengan kata baku, per kata."""
    return " ".join(slang.get(word, word) for word in text.split())


def near_duplicate_groups(texts: list[str], threshold: float = 0.9) -> np.ndarray:
    """Beri id grup yang sama pada teks yang kemiripan char TF-IDF-nya di atas threshold."""
    vectors = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1).fit_transform(texts)
    neighbors = NearestNeighbors(n_neighbors=min(6, len(texts)), metric="cosine").fit(vectors)
    distance, index = neighbors.kneighbors(vectors)
    parent = list(range(len(texts)))

    def root(i: int) -> int:
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for i, (row_dist, row_idx) in enumerate(zip(distance, index)):
        for dist, j in zip(row_dist, row_idx):
            if dist <= 1 - threshold:
                parent[root(i)] = root(int(j))
    return np.array([root(i) for i in range(len(texts))])


def adversarial_auc(texts_a: list[str], texts_b: list[str]) -> float:
    """AUC model yang membedakan dua kumpulan teks. Dekat 0,5 berarti tidak ada shift."""
    texts = list(texts_a) + list(texts_b)
    target = np.array([0] * len(texts_a) + [1] * len(texts_b))
    vectors = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit_transform(texts)
    model = LogisticRegression(max_iter=1000, class_weight="balanced")
    scores = cross_val_predict(model, vectors, target, cv=3, method="predict_proba")[:, 1]
    return float(roc_auc_score(target, scores))


def text_profile(texts: pd.Series) -> pd.Series:
    """Ringkasan kualitas teks untuk EDA."""
    words = texts.astype(str).str.split().str.len()
    return pd.Series(
        {
            "baris": len(texts),
            "kosong": int((words == 0).sum()),
            "duplikat_persis_%": round(100 * texts.duplicated().mean(), 1),
            "kata_median": float(words.median()),
            "kata_p95": float(words.quantile(0.95)),
            "kata_maks": int(words.max()),
            "ada_url_%": round(100 * texts.astype(str).str.contains(r"https?://").mean(), 1),
            "ada_emoji_%": round(
                100 * texts.astype(str).str.contains(r"[\U0001F300-\U0001FAFF]").mean(), 1
            ),
        }
    )

**Metrik, interval noise, dan pencatat hasil.** Metrik diuji dulu pada contoh yang jawabannya
dihitung manual. `record` mencatat skor OOF dan skor holdout untuk tiap pendekatan.

In [6]:
from sklearn.metrics import f1_score  # noqa: E402
from sklearn.model_selection import StratifiedGroupKFold  # noqa: E402


def macro_f1(y_true, y_pred) -> float:
    return float(f1_score(y_true, y_pred, average="macro", zero_division=0))


assert abs(macro_f1([0, 0, 1, 1], [0, 1, 1, 1]) - (2 / 3 + 0.8) / 2) < 1e-9


def bootstrap_interval(y_true, y_pred, score_fn=macro_f1, n_boot: int = 300) -> str:
    """Interval 95%. Selisih di dalam interval ini belum tentu perbaikan nyata."""
    rng = np.random.default_rng(SEED)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    scores = []
    for _ in range(n_boot):
        pick = rng.integers(0, len(y_true), len(y_true))
        scores.append(score_fn(y_true[pick], y_pred[pick]))
    return f"{np.percentile(scores, 2.5):.3f}-{np.percentile(scores, 97.5):.3f}"


def make_folds(target, groups, n_folds: int) -> np.ndarray:
    """StratifiedGroupKFold: duplikat tidak lintas fold, proporsi kelas terjaga."""
    splitter = StratifiedGroupKFold(n_folds, shuffle=True, random_state=SEED)
    folds = np.zeros(len(target), dtype=int)
    for fold, (_, val_idx) in enumerate(splitter.split(np.zeros(len(target)), target, groups)):
        folds[val_idx] = fold
    return folds


results: list[dict] = []


def record(name: str, oof_pred, holdout_pred, score_fn=macro_f1) -> None:
    """Catat skor OOF dan holdout. Holdout hanya untuk laporan, bukan untuk memilih model."""
    results.append(
        {
            "model": name,
            "oof": round(score_fn(y, oof_pred), 4),
            "oof_ci95": bootstrap_interval(y, oof_pred, score_fn),
            "holdout": round(score_fn(y_holdout, holdout_pred), 4),
        }
    )
    print(results[-1])

**Training transformer.** Satu fungsi untuk klasifikasi (`problem="ce"`), multi-label (`"bce"`),
dan regresi (`"mse"`). Loop-nya ditulis sendiri supaya mudah dibaca dan outputnya bersih.

In [7]:
from torch.utils.data import DataLoader, TensorDataset  # noqa: E402
from transformers import (  # noqa: E402
    AutoModelForSequenceClassification,
    AutoTokenizer as _SeqTokenizer,
    get_linear_schedule_with_warmup,
)


def _encode(tokenizer, texts, pairs, max_len):
    args = (list(texts), list(pairs)) if pairs is not None else (list(texts),)
    tokens = tokenizer(
        *args, padding="max_length", truncation=True, max_length=max_len, return_tensors="pt"
    )
    return tokens["input_ids"], tokens["attention_mask"]


def _loss_fn(problem, weights):
    if problem == "ce":
        return torch.nn.CrossEntropyLoss(weight=weights)
    # untuk multi-label, class_weights berperan sebagai pos_weight per label
    return (
        torch.nn.BCEWithLogitsLoss(pos_weight=weights) if problem == "bce" else torch.nn.MSELoss()
    )


def _to_output(logits, problem):
    if problem == "ce":
        return torch.softmax(logits.float(), dim=1)
    if problem == "bce":
        return torch.sigmoid(logits.float())
    return logits.float().squeeze(-1)


@torch.inference_mode()
def _predict(model, ids, mask, problem, batch_size):
    model.eval()
    chunks = []
    for start in range(0, len(ids), batch_size):
        batch_ids = ids[start : start + batch_size].to(DEVICE)
        batch_mask = mask[start : start + batch_size].to(DEVICE)
        with torch.autocast("cuda", enabled=DEVICE == "cuda"):
            logits = model(input_ids=batch_ids, attention_mask=batch_mask).logits
        chunks.append(_to_output(logits, problem).cpu())
    return torch.cat(chunks).numpy()


def train_predict(
    model_name,
    train_texts,
    train_y,
    predict_sets,
    problem="ce",
    num_labels=None,
    class_weights=None,
    train_pairs=None,
    epochs=3,
    lr=2e-5,
    batch_size=32,
    max_len=128,
    seed=42,
):
    """Latih satu model, lalu kembalikan prediksi untuk tiap set di predict_sets.

    predict_sets: daftar teks, atau daftar pasangan (teks, teks_pair) untuk tugas dua teks.
    """
    seed_everything(seed)
    tokenizer = _SeqTokenizer.from_pretrained(MODEL_DIR / model_name)
    num_labels = num_labels or (int(np.max(train_y)) + 1 if problem == "ce" else 1)
    if problem == "bce":
        num_labels = np.asarray(train_y).shape[1]
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_DIR / model_name, num_labels=num_labels, ignore_mismatched_sizes=True
    ).to(DEVICE)

    ids, mask = _encode(tokenizer, train_texts, train_pairs, max_len)
    target = torch.tensor(np.asarray(train_y), dtype=torch.long if problem == "ce" else torch.float)
    if problem == "mse":
        target = target.view(-1, 1)
    loader = DataLoader(
        TensorDataset(ids, mask, target),
        batch_size=batch_size,
        shuffle=True,
        drop_last=len(ids) > batch_size,
    )
    weights = (
        None if class_weights is None else torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
    )
    loss_fn = _loss_fn(problem, weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = max(1, epochs * len(loader))
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")

    model.train()
    for _ in range(epochs):
        for batch_ids, batch_mask, batch_y in loader:
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=DEVICE == "cuda"):
                logits = model(
                    input_ids=batch_ids.to(DEVICE), attention_mask=batch_mask.to(DEVICE)
                ).logits
                loss = loss_fn(logits, batch_y.to(DEVICE))
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

    outputs = []
    for item in predict_sets:
        texts, pairs = item if isinstance(item, tuple) else (item, None)
        pred_ids, pred_mask = _encode(tokenizer, texts, pairs, max_len)
        outputs.append(_predict(model, pred_ids, pred_mask, problem, batch_size * 2))
    del model, optimizer, scaler
    torch.cuda.empty_cache()
    return outputs

**Pembaca IndoSum.** Tiap artikel berupa paragraf, kalimat, dan token. `load_indosum` meratakannya menjadi
daftar kalimat, ringkasan, dan penanda kalimat mana yang masuk ringkasan.

In [8]:
def load_indosum(file_name: str, n_rows: int) -> pd.DataFrame:
    frame = pd.read_json(find_file(file_name), lines=True).head(n_rows)

    def flatten(paragraphs) -> list[str]:
        return [" ".join(tokens) for paragraph in paragraphs for tokens in paragraph]

    frame["sentences"] = frame["paragraphs"].map(flatten)
    frame["article"] = frame["sentences"].map(" ".join)
    frame["summary_text"] = frame["summary"].map(lambda s: " ".join(" ".join(t) for t in s))
    frame["first_summary"] = frame["summary"].map(lambda s: " ".join(s[0]) if len(s) else "")
    frame["labels"] = frame["gold_labels"].map(
        lambda g: [flag for paragraph in g for flag in paragraph]
    )
    return frame[["category", "sentences", "article", "summary_text", "first_summary", "labels"]]

## 4. EDA

Muat artikel. Tiap artikel menghasilkan satu pasangan, jadi jumlah pasangan sama dengan `N_ROWS`.

In [9]:
articles = load_indosum("test.01.jsonl", N_ROWS)
articles = pd.concat([articles, load_indosum("dev.01.jsonl", N_ROWS)], ignore_index=True).head(
    N_ROWS
)
articles = articles[articles["first_summary"].str.split().str.len() >= 6].reset_index(drop=True)
print(f"artikel: {len(articles)}")
print(articles["category"].value_counts().to_dict())
print("contoh judul:", articles["first_summary"][0][:120])
text_profile(articles["first_summary"])

artikel: 4498
{'tajuk utama': 1734, 'olahraga': 1140, 'showbiz': 594, 'teknologi': 566, 'hiburan': 430, 'inspirasi': 34}
contoh judul: Eman Ahmed Abd El Aty memiliki berat badan mencapai 500 kilogram sebelum menjalankan operasi di Mumbai Maret lalu dimana


baris                4498.0
kosong                  0.0
duplikat_persis_%       0.0
kata_median            22.0
kata_p95               38.0
kata_maks              69.0
ada_url_%               0.0
ada_emoji_%             0.0
dtype: float64

**Temuan dan keputusan**

| Temuan | Keputusan |
|---|---|
| Tidak ada label asli | Buat tiga jenis negatif: artikel lain, entitas ditukar, angka diubah |
| Satu artikel menghasilkan beberapa pasangan | Fold berdasarkan artikel |
| Isi artikel panjang | Isi dipotong karena batas token model |

## 5. Preprocessing

Separuh pasangan benar, separuh negatif dengan tiga jenis yang sulitnya berbeda: artikel lain di kategori yang
sama, entitas ditukar, dan angka diubah. Isi artikel dipotong, karena token model terbatas.

In [10]:
ENTITY_RE = re.compile(r"\b[A-Z][a-z]{2,}\b")
NUMBER_RE = re.compile(r"\b\d+\b")


def swap_entity(title: str, pool: list[str], rng) -> str | None:
    found = ENTITY_RE.findall(title)
    if not found:
        return None
    return title.replace(str(rng.choice(found)), str(rng.choice(pool)), 1)


def change_number(title: str, rng) -> str | None:
    found = NUMBER_RE.findall(title)
    if not found:
        return None
    number = str(rng.choice(found))
    return title.replace(number, str(int(number) + int(rng.integers(1, 9))), 1)


def build_pairs() -> pd.DataFrame:
    rng = np.random.default_rng(SEED)
    entity_pool = sorted(set(ENTITY_RE.findall(" ".join(articles["first_summary"]))))
    rows = []
    for i, row in articles.iterrows():
        title, kind = row["first_summary"], "asli"
        if rng.random() < NEGATIVE_SHARE:
            choice = rng.choice(["artikel_lain", "entitas", "angka"])
            if choice == "entitas":
                changed = swap_entity(title, entity_pool, rng)
            elif choice == "angka":
                changed = change_number(title, rng)
            else:
                others = articles[(articles["category"] == row["category"]) & (articles.index != i)]
                changed = (
                    others["first_summary"].sample(1, random_state=int(rng.integers(1e9))).iloc[0]
                )
            if changed:
                title, kind = changed, str(choice)
        rows.append(
            {
                "article_id": i,
                "title": title,
                "body": " ".join(row["article"].split()[:BODY_WORDS]),
                "jenis": kind,
                "label": int(kind == "asli"),
            }
        )
    return pd.DataFrame(rows)


pairs = build_pairs()
print(pairs["jenis"].value_counts().to_dict())
holdout_ids = np.random.default_rng(SEED).choice(
    len(articles), int(0.2 * len(articles)), replace=False
)
is_holdout = pairs["article_id"].isin(holdout_ids)
train, holdout = pairs[~is_holdout].reset_index(drop=True), pairs[is_holdout].reset_index(drop=True)
y, y_holdout = train["label"].to_numpy(), holdout["label"].to_numpy()
print(f"latih={len(train)} | holdout={len(holdout)}")

{'asli': 2641, 'artikel_lain': 782, 'entitas': 754, 'angka': 321}
latih=3599 | holdout=899


## 6. Validasi

Fold berdasarkan artikel, jadi artikel yang sama tidak pernah ada di latih dan validasi sekaligus. Metriknya
macro-F1. Semua fitur dihitung per pasangan dan tidak memakai label.

In [11]:
folds = make_folds(y, train["article_id"], SETTINGS["folds"])
print("baris per fold:", np.bincount(folds).tolist())

baris per fold: [720, 720, 720, 720, 719]


## 7. Baseline

Tebak mayoritas, lalu fitur relasi: seberapa banyak kata, entitas, dan angka judul yang muncul di isi. Fitur ini
sering sangat kuat dan tetap jalan tanpa model dari luar.

In [12]:
def relation_features(frame: pd.DataFrame) -> np.ndarray:
    rows = []
    for title, body in zip(frame["title"], frame["body"]):
        words = set(title.lower().split())
        body_lower = body.lower()
        body_set = set(body_lower.split())
        entities, numbers = ENTITY_RE.findall(title), NUMBER_RE.findall(title)
        rows.append(
            [
                len(words & body_set) / max(len(words), 1),
                np.mean([e in body for e in entities]) if entities else 1.0,
                np.mean([n in body for n in numbers]) if numbers else 1.0,
                float(all(e in body for e in entities)),
                float(all(n in body for n in numbers)),
                len(title.split()),
            ]
        )
    return np.array(rows, dtype=float)


majority = np.bincount(y).argmax()
record("mayoritas", np.full(len(y), majority), np.full(len(y_holdout), majority))
x_train, x_holdout = relation_features(train), relation_features(holdout)
oof_lr, hold_lr = np.zeros(len(y)), np.zeros(len(y_holdout))
for fold in range(SETTINGS["folds"]):
    fit, val = folds != fold, folds == fold
    model = LogisticRegression(max_iter=2000, class_weight="balanced").fit(x_train[fit], y[fit])
    oof_lr[val] = model.predict_proba(x_train[val])[:, 1]
    hold_lr += model.predict_proba(x_holdout)[:, 1] / SETTINGS["folds"]
record("fitur_relasi", (oof_lr >= 0.5).astype(int), (hold_lr >= 0.5).astype(int))

{'model': 'mayoritas', 'oof': 0.3703, 'oof_ci95': '0.364-0.376', 'holdout': 0.3687}


{'model': 'fitur_relasi', 'oof': 0.9236, 'oof_ci95': '0.914-0.932', 'holdout': 0.919}


## 8. Model Utama

Cross-encoder: judul dan isi masuk bersamaan, jadi model bisa membandingkan keduanya.

In [13]:
oof_cross, hold_cross = oof_lr.copy(), hold_lr.copy()
with stage("cross_encoder"):
    if USE_PRETRAINED:
        oof_cross, hold_cross = np.zeros(len(y)), np.zeros(len(y_holdout))
        for model_name, seed in SETTINGS["model_runs"]:
            for fold in range(SETTINGS["folds"]):
                fit, val = folds != fold, folds == fold
                pred_val, pred_hold = train_predict(
                    model_name,
                    train.loc[fit, "title"].tolist(),
                    y[fit],
                    [
                        (train.loc[val, "title"].tolist(), train.loc[val, "body"].tolist()),
                        (holdout["title"].tolist(), holdout["body"].tolist()),
                    ],
                    num_labels=2,
                    train_pairs=train.loc[fit, "body"].tolist(),
                    epochs=SETTINGS["epochs"],
                    seed=seed + fold,
                    batch_size=16,
                    max_len=MAX_LEN,
                )
                oof_cross[val] += pred_val[:, 1] / len(SETTINGS["model_runs"])
                hold_cross += pred_hold[:, 1] / (SETTINGS["folds"] * len(SETTINGS["model_runs"]))
        record("cross_encoder", (oof_cross >= 0.5).astype(int), (hold_cross >= 0.5).astype(int))

{'model': 'cross_encoder', 'oof': 0.9174, 'oof_ci95': '0.908-0.925', 'holdout': 0.9127}
[waktu] cross_encoder: 1452 detik


## 9. Post-processing

Stacking cross-encoder dengan fitur relasi, karena keduanya menangkap kesalahan berbeda. Stacker dan threshold
difit di dalam fold, jadi skornya jujur.

In [14]:
def best_threshold(scores, target) -> float:
    grid = np.arange(0.2, 0.81, 0.01)
    return float(grid[int(np.argmax([macro_f1(target, scores >= t) for t in grid]))])


x_stack = np.column_stack([oof_cross, x_train])
x_stack_hold = np.column_stack([hold_cross, x_holdout])
oof_stack, hold_stack = np.zeros(len(y)), np.zeros(len(y_holdout))
oof_pred = np.zeros(len(y), dtype=int)
for fold in range(SETTINGS["folds"]):
    fit, val = folds != fold, folds == fold
    stacker = LogisticRegression(max_iter=2000).fit(x_stack[fit], y[fit])
    oof_stack[val] = stacker.predict_proba(x_stack[val])[:, 1]
    oof_pred[val] = oof_stack[val] >= best_threshold(
        stacker.predict_proba(x_stack[fit])[:, 1], y[fit]
    )
    hold_stack += stacker.predict_proba(x_stack_hold)[:, 1] / SETTINGS["folds"]
hold_pred = (hold_stack >= best_threshold(oof_stack, y)).astype(int)
record("stacking_plus_threshold", oof_pred, hold_pred)

{'model': 'stacking_plus_threshold', 'oof': 0.9446, 'oof_ci95': '0.938-0.952', 'holdout': 0.9533}


## 10. Post-analysis

**Tabel perbandingan.**

In [15]:
result_table = pd.DataFrame(results)
result_table.to_csv(RESULT_DIR / "perbandingan.csv", index=False)
result_table

,model,oof,oof_ci95,holdout
0,mayoritas,0.3703,0.364-0.376,0.3687
1,fitur_relasi,0.9236,0.914-0.932,0.9190
2,cross_encoder,0.9174,0.908-0.925,0.9127
3,stacking_plus_threshold,0.9446,0.938-0.952,0.9533


**Skor menurut jenis pasangan.** Ini yang paling penting: jenis mana yang berhasil dan gagal dideteksi.

In [16]:
holdout.assign(pred=hold_pred, benar=hold_pred == y_holdout).groupby("jenis")["benar"].agg(
    ["size", "mean"]
).round(3)

,size,mean
jenis,,
angka,70,0.886
artikel_lain,163,1.000
asli,525,0.947
entitas,141,0.965


**Salah prediksi.**

In [17]:
errors = holdout.assign(pred=hold_pred)
errors[errors["label"] != errors["pred"]][["title", "jenis", "label"]].head(8)

,title,jenis,label
20,Pkembangan pesat teknologi artificial intelligence ( AI ) dibuktikan oleh situs bernam...,asli,1
108,Bek Barcelona Javier Mascherano akan hengkang ke klub liga Tiongkok Hebei China Fortun...,angka,0
161,"Aktris Syifa Hadju bermain dalam film ' Ayat - ayat Cinta 2 ' , yang telah tayang di b...",asli,1
197,"Timnas Indonesia beruntung , dikarenakan bek Que Ngoc Hai tidak akan memperkuat Timn...",asli,1
207,John Tery dan Gary Cahill mengunjungi Ryan Mason di rumah sakit setelah sang gelandang...,asli,1
211,"Seorang perempuan bercadar di Barcelona , Spanyol , menjadi korban cacian dan tendanga...",asli,1
236,"Bila saja Song Joong Ki dan Song Hye Kyo dipertemukan dua belas tahun yang lalu , mung...",asli,1
246,Kabar terkini datang dari pasangan calon gubernur dan wakil gubernur DKI Jakarta Anies...,asli,1


## 11. Keluaran

File untuk dikumpulkan dan ringkasan run.

In [18]:
pd.DataFrame({"id": holdout.index, "label": hold_pred}).to_csv(
    RESULT_DIR / "submission.csv", index=False
)
run_summary(
    "konsistensi",
    len(pairs),
    {r["model"]: {"oof": r["oof"], "holdout": r["holdout"]} for r in results},
    needs_gpu=USE_PRETRAINED,
)

RINGKASAN_RUN task=konsistensi mode=penuh n_rows=4498 total_detik=1476 gpu=True
{
 "mayoritas": {
  "oof": 0.3703,
  "holdout": 0.3687
 },
 "fitur_relasi": {
  "oof": 0.9236,
  "holdout": 0.919
 },
 "cross_encoder": {
  "oof": 0.9174,
  "holdout": 0.9127
 },
 "stacking_plus_threshold": {
  "oof": 0.9446,
  "holdout": 0.9533
 }
}


**Poin untuk presentasi**

- Stacking terbaik: holdout 0,9533, sedangkan fitur relasi 0,919 dan cross-encoder 0,9127.
- Fitur sederhana sudah mengalahkan cross-encoder. Keduanya saling melengkapi.
- Akurasi per jenis: artikel lain 1,0, entitas 0,965, asli 0,947, angka 0,886.
- Skor pada keluhan MBG asli bisa jauh lebih rendah, karena kesalahan aslinya lebih halus.